# FastText — trigrams at 700k / 1.5M / 3.6M

The selected configuration is held fixed across the three sizes, so sample size is the only variable in the final learning curve.

Source CSVs remain on Windows, while FastText corpora and models are stored on the WSL Linux filesystem and reused when their cache stamps match.


## Setup


In [1]:
# STEP 0 — CONFIG
from pathlib import Path
import gc
import json
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import fasttext
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Source CSVs stay on the Windows disk (same files the baseline used).
# ============================================================
# REPRODUCIBILITY -- REPLACE THIS WITH YOUR OWN DATA PATH.
# Point it at the folder holding the Amazon Polarity CSVs.
# That folder must contain: train.csv and test.csv
# Everything else in this notebook is relative to the notebook
# directory (artifacts/, results_csv/) and needs no editing.
# ============================================================
DATA_DIR = Path("/mnt/c/Users/bsarv/Fake Desktop/amz sentiment analysis/archive")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"

# fastText corpora + models live on the Linux disk.
# through /mnt/c is an order of magnitude slower.
FT_DIR = Path.home() / "ft"
FT_DIR.mkdir(exist_ok=True)

RESULTS_DIR = Path("results_csv")
RESULTS_DIR.mkdir(exist_ok=True)

COLUMN_NAMES = ["label", "title", "text"]

# Cache control. Everything below is skipped when a matching artifact is on disk.
FORCE_REBUILD   = False   # ignore cached corpora
FORCE_REFIT     = False   # ignore cached models
RUN_EXPERIMENTS = False   # noise floor + minCount grid (steps 11, 12)

# Identical to the baseline so the two curves are comparable.
SEED = 42
CHUNK = 100_000
N_CHUNKS = 36
TOTAL_ROWS = 3_600_000
VAL_SIZE = 0.2
# SIZES = [200_000, 500_000] # for checking
SIZES = [700_000, 1_500_000, TOTAL_ROWS] 

# Treat great and GREAT as the same.
LOWERCASE = True
# Tokenize punctuation with its word.
PAD_PUNCT = False
# Combine title+" "+body.
USE_TITLE = True

# fastText defaults for the first run.
FT_PARAMS = dict(
    lr=0.07, # learning rate
    epoch=5, # times we go through training dataset
    wordNgrams=3, # keeping it at 2 as baseline also bi-gram
    dim=10, # vec size
    minCount=9, # prune rare words; 12x smaller vocab at 0.00 pp cost (see step 12)
    bucket=10_000_000, # I believe this the number of vectors we have in the bank to assinge to n-gram words
    loss="softmax",
    thread=8, # cpu
    seed=42, #reproduciblity
    verbose=2,
)


# --- cache stamps -------------------------------------------------------------
# Every cached artifact gets a sidecar <name>.json holding the config that made it.
# Mismatch -> rebuild. Without this, editing LOWERCASE or lr silently reuses a stale file.
def _sidecar(path):
    return path.with_suffix(path.suffix + ".json")

def stamp_ok(path, stamp):
    s = _sidecar(path)
    if not (path.exists() and s.exists()):
        return False
    return json.loads(s.read_text()) == json.loads(json.dumps(stamp, sort_keys=True))

def stamp_write(path, stamp):
    _sidecar(path).write_text(json.dumps(stamp, sort_keys=True))

# corpus-side: these change the bytes on disk
CORPUS_STAMP = dict(lowercase=LOWERCASE, pad_punct=PAD_PUNCT, use_title=USE_TITLE,
                    val_size=VAL_SIZE, seed=SEED)

print(f"fastText {fasttext.__file__}")
print(f"corpora -> {FT_DIR}")

fastText /home/bsarv/.venvs/fasttext/lib/python3.12/site-packages/fasttext/__init__.py
corpora -> /home/bsarv/ft


## Cache control

Each cached corpus and final model has a JSON sidecar containing the configuration that produced it. `FORCE_REBUILD` and `FORCE_REFIT` bypass those caches; `ADOPT_EXISTING` is only for one-time stamping of older corpora.


In [2]:
# STEP 0b — ADOPT EXISTING CORPORA (one-time)
# Corpora built before stamping exist but have no sidecar, so they would rebuild.
# Flip to True for one run to stamp them with the CURRENT settings, then flip back.
# Leaving this True defeats the stamp: it would bless whatever is on disk every run.
ADOPT_EXISTING = False

if ADOPT_EXISTING:
    for f in list(FT_DIR.glob("train_*.txt")) + list(FT_DIR.glob("val_*.txt")) + [FT_DIR / "test.txt"]:
        if f.exists():
            stamp_write(f, CORPUS_STAMP)
            print(f"stamped {f.name}")


## 1. Data preparation

The headerless CSV is read as `[label, title, text]`. Title and body are combined, whitespace is normalized, text is lowercased, and each sample is split 80/20 with label stratification.


In [3]:
# STEP 1 — INGEST
"""preping data: combining body and title"""
def ingest(path, n=None, chunks=False):
    """CSV -> DataFrame[label, combined]. n=None reads the whole file."""
    t0 = time.perf_counter()

    df = pd.read_csv(
        path,
        header=None,
        names=COLUMN_NAMES,
        dtype={"label": "int8"},
        keep_default_na=False,   # empty title stays "", not NaN
        nrows=n,
    )

    if USE_TITLE:
        df["combined"] = (df["title"] + " " + df["text"]).str.strip()
    else:
        df["combined"] = df["text"].str.strip()

    df = df[["label", "combined"]]

    print(
        f"{len(df):,} rows in {time.perf_counter()-t0:.1f}s  "
        f"balance={np.bincount(df['label'].to_numpy())[1:]}  "
        f"memory={df.memory_usage(deep=True).sum()/1e6:.0f} MB"
    )
    return df


# guarded: the notebook runs from cached corpora on a machine without the source CSVs
if TRAIN_PATH.exists():
    peek = ingest(TRAIN_PATH, n=20_000)
    display(peek.head(3))
else:
    print("source CSVs absent — running from cached corpora")

20,000 rows in 0.3s  balance=[ 9743 10257]  memory=10 MB


,label,combined
0,2,Stuning even for the non-gamer This sound trac...
1,2,The best soundtrack ever to anything. I'm read...
2,2,Amazing! This soundtrack is my favorite music ...


In [4]:
# STEP 2 — NORMALIZE
# cleaning and processing punctuation
_WS    = re.compile(r"\s+")
_PUNCT = re.compile(r"([.,!?;:()\"'])")

def normalize(s):
    """Row-wise cleaning. Must be identical for train, val and test."""
    # s = s.str.replace("__label__", "label", regex=False)  # can't forge a label: scanned code, do not need to clean this
    if PAD_PUNCT:
        s = s.str.replace(_PUNCT, r" \1 ", regex=True)
    s = s.str.replace(_WS, " ", regex=True)               # \n \r \t and runs -> one space
    if LOWERCASE:
        s = s.str.lower()
    return s.str.strip()


def prepare(df):
    """normalize + drop rows that end up empty."""
    df = df.assign(combined=normalize(df["combined"]))
    n0 = len(df)
    df = df[df["combined"].str.len() > 0]
    if n0 != len(df):
        print(f"dropped {n0-len(df):,} empty rows")
    return df

In [5]:
# STEP 3 — SPLIT
#split to test & val
def split(df, seed=SEED):
    return train_test_split(
        df, test_size=VAL_SIZE, random_state=seed, stratify=df["label"]
    )

In [6]:
# STEP 3b — CHECK (on peek)
if TRAIN_PATH.exists():
    p = prepare(peek)
    tr, va = split(p)
    print(len(tr), len(va))
    print(p["combined"].str.contains(r"[\n\r\t]").sum(), "rows still holding whitespace chars")
    print(repr(p.iloc[1]["combined"][:200]))


16000 4000
0 rows still holding whitespace chars
"the best soundtrack ever to anything. i'm reading a lot of reviews saying that this is the best 'game soundtrack' and i figured that i'd write a review to disagree a bit. this in my opinino is yasunor"


## 2. FastText corpora

Each row is written as `__label__<class> text`. Training and validation files are created for every sample size, while the official test set is prepared once without splitting.


In [7]:
# STEP 4 — WRITE
def write_ft(df, path):
    """DataFrame[label, combined] -> fastText supervised text file.

    Writes to .tmp and renames. An interrupted build never leaves a
    valid-looking partial file under the real name, so "exists" is a
    trustworthy cache signal.
    """
    t0 = time.perf_counter()
    tmp = path.with_suffix(path.suffix + ".tmp")
    lines = "__label__" + df["label"].astype(str) + " " + df["combined"]
    with open(tmp, "w", encoding="utf-8", newline="\n") as f:
        for i in range(0, len(lines), CHUNK):
            f.write("\n".join(lines.iloc[i:i+CHUNK]))
            f.write("\n")
    os.replace(tmp, path)
    mb = path.stat().st_size / 1e6 #calc the size in megabytes mb
    print(f"{path.name}: {len(lines):,} lines, {mb:.0f} MB, {time.perf_counter()-t0:.1f}s")


In [8]:
# STEP 5 — BUILD TRAIN/VAL
def build(size):
    tr_p = FT_DIR / f"train_{size}.txt"
    va_p = FT_DIR / f"val_{size}.txt"
    if not FORCE_REBUILD and stamp_ok(tr_p, CORPUS_STAMP) and stamp_ok(va_p, CORPUS_STAMP):
        print(f"{size:,}: corpora cached, skipped")
        return

    df = prepare(ingest(TRAIN_PATH, n=size))
    tr, va = split(df)
    write_ft(tr, tr_p); stamp_write(tr_p, CORPUS_STAMP)
    write_ft(va, va_p); stamp_write(va_p, CORPUS_STAMP)
    del df, tr, va; gc.collect()

for size in SIZES:
    build(size)


700,000: corpora cached, skipped
1,500,000: corpora cached, skipped
3,600,000: corpora cached, skipped


In [9]:
# STEP 6 — BUILD TEST
# test.csv goes through the same prepare() path as train, but is never split — it is the held-out set, used whole.
test_path = FT_DIR / "test.txt"
if FORCE_REBUILD or not stamp_ok(test_path, CORPUS_STAMP):
    te = prepare(ingest(TEST_PATH))
    write_ft(te, test_path); stamp_write(test_path, CORPUS_STAMP)
    del te; gc.collect()
else:
    print("test.txt cached, skipped")


test.txt cached, skipped


## 3. Training and validation

The same selected FastText configuration is trained or reloaded for each size and evaluated on that size's validation file. Results are retained in CSV so completed runs survive a fresh kernel.


In [10]:
# STEP 8 — EVAL (val)
def load_ft(path):
    """fastText .txt -> (y, texts). Reads the file the model itself reads, so nothing is normalized twice."""
    y, x = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            lab, _, txt = line.rstrip("\n").partition(" ")
            y.append(int(lab.removeprefix("__label__")))
            x.append(txt)
    return np.array(y, dtype="int8"), x


def evaluate(model, path):
    """-> (y_true, y_pred, predict_seconds)."""
    y, x = load_ft(path)
    t0 = time.perf_counter()
    labs, _ = model.predict(x)
    secs = time.perf_counter() - t0
    yhat = np.array([int(l[0].removeprefix("__label__")) for l in labs], dtype="int8")
    return y, yhat, secs


# y_val, yhat_val, pred_secs = evaluate(model, val_path)
# acc = accuracy_score(y_val, yhat_val)
# f1 = f1_score(y_val, yhat_val, average="macro")
# print(f"{SIZE:,}: val acc={acc:.4f}  macro F1={f1:.4f}  ({len(y_val):,} rows in {pred_secs:.1f}s)")
# print(classification_report(y_val, yhat_val, digits=4))
# print(confusion_matrix(y_val, yhat_val))

# same columns as baseline_full_results.csv so the two curves stack; keyed by size, so a re-run replaces its row
try:
    ft_results
except NameError:
    ft_results = {}
# ft_results[SIZE] = dict(
#     sample_size=SIZE,
#     val_rows=len(y_val),
#     vocab=len(model.words),
#     validation_accuracy=acc,
#     validation_f1=f1,
#     fit_seconds=round(train_secs, 1),
#     predict_seconds=round(pred_secs, 1),
# )
# pd.DataFrame(ft_results.values())

In [11]:
# STEP 9 — SWEEP (fit on train_{size}, score on val_{size})
TUNE_SIZE = 700_000
FINAL = dict(lr=0.07, epoch=5, dim=10, wordNgrams=3, minCount=9, bucket=10_000_000)
SWEEP_CSV = RESULTS_DIR / "fasttext_epoch_sweep.csv"

MODEL_STAMP = {k: v for k, v in {**FT_PARAMS, **FINAL}.items() if k != "verbose"}

# keyed by (size, config). Seeded from the CSV so a fresh kernel does not
# overwrite rows from earlier sessions.
try:
    ft_sweep
except NameError:
    ft_sweep = {}
    if SWEEP_CSV.exists():
        for r in pd.read_csv(SWEEP_CSV).to_dict("records"):
            ft_sweep[(r["sample_size"], r["config"])] = r


def fit_or_load(size, over):
    """-> (model, fit_secs). fit_secs is None when loaded from disk.

    Only FINAL is cached — sweep variants are fit fresh so they cannot
    overwrite final_{size}.bin.
    """
    train_path = FT_DIR / f"train_{size}.txt"
    if over == FINAL:
        bin_path = FT_DIR / f"final_{size}.bin"
        if not FORCE_REFIT and stamp_ok(bin_path, MODEL_STAMP):
            return fasttext.load_model(str(bin_path)), None
        t0 = time.perf_counter()
        model = fasttext.train_supervised(input=str(train_path), **{**FT_PARAMS, **FINAL})
        secs = time.perf_counter() - t0
        model.save_model(str(bin_path)); stamp_write(bin_path, MODEL_STAMP)
        return model, secs

    t0 = time.perf_counter()
    model = fasttext.train_supervised(input=str(train_path), **{**FT_PARAMS, **over})
    return model, time.perf_counter() - t0


def sweep(size, over):
    params = {**FT_PARAMS, **over}
    label  = ", ".join(f"{k}={v}" for k, v in sorted(over.items())) or "defaults"
    key    = (size, label)

    model, fit_secs = fit_or_load(size, over)
    y, yhat, pred_secs = evaluate(model, FT_DIR / f"val_{size}.txt")
    acc = accuracy_score(y, yhat)
    f1  = f1_score(y, yhat, average="macro")
    tag = "cached" if fit_secs is None else f"fit={fit_secs:.1f}s"
    print(f"{size:,} | {label}: acc={acc:.4f} f1={f1:.4f} {tag}")

    if fit_secs is None and key in ft_sweep:
        # loaded model: keep the fit_seconds measured by the run that trained it
        ft_sweep[key].update(validation_accuracy=acc, validation_f1=f1,
                             predict_seconds=round(pred_secs, 1))
    else:
        ft_sweep[key] = dict(
            sample_size=size, config=label,
            lr=params["lr"], dim=params["dim"], epoch=params["epoch"],
            wordNgrams=params["wordNgrams"], minCount=params["minCount"], bucket=params["bucket"],
            val_rows=len(y), vocab=len(model.words),
            validation_accuracy=acc, validation_f1=f1,
            fit_seconds=None if fit_secs is None else round(fit_secs, 1),
            predict_seconds=round(pred_secs, 1),
        )
    pd.DataFrame(ft_sweep.values()).to_csv(SWEEP_CSV, index=False)
    del model; gc.collect()


for size in SIZES:
    sweep(size, FINAL)

pd.DataFrame(ft_sweep.values()).sort_values("validation_accuracy", ascending=False)


700,000 | bucket=10000000, dim=10, epoch=5, lr=0.07, minCount=9, wordNgrams=3: acc=0.9321 f1=0.9321 cached
1,500,000 | bucket=10000000, dim=10, epoch=5, lr=0.07, minCount=9, wordNgrams=3: acc=0.9378 f1=0.9378 cached
3,600,000 | bucket=10000000, dim=10, epoch=5, lr=0.07, minCount=9, wordNgrams=3: acc=0.9434 f1=0.9434 cached


,sample_size,config,lr,dim,epoch,bucket,wordNgrams,val_rows,vocab,validation_accuracy,validation_f1,fit_seconds,predict_seconds,minCount
19,3600000,"epoch=5, lr=0.07, wordNgrams=3",0.07,10,5,10000000,3,720000,3629672,0.943417,0.943417,93.7,70.3,NaN
26,3600000,"bucket=10000000, dim=10, epoch=5, lr=0.07, min...",0.07,10,5,10000000,3,720000,270790,0.943404,0.943404,75.3,40.8,9.0
11,3600000,"epoch=5, lr=0.07",0.07,10,5,10000000,2,720000,3629672,0.940712,0.940712,54.5,32.3,NaN
25,1500000,"bucket=10000000, dim=10, epoch=5, lr=0.07, min...",0.07,10,5,10000000,3,300000,155463,0.937830,0.937828,32.8,11.5,9.0
18,1500000,"epoch=5, lr=0.07, wordNgrams=3",0.07,10,5,10000000,3,300000,1940502,0.937600,0.937598,34.2,10.8,NaN
10,1500000,"epoch=5, lr=0.07",0.07,10,5,10000000,2,300000,1940502,0.935407,0.935405,22.1,5.3,NaN
15,700000,wordNgrams=3,0.10,10,5,10000000,3,140000,1116466,0.932836,0.932828,26.0,4.7,NaN
20,700000,"epoch=5, lr=0.07, minCount=1, wordNgrams=3",0.07,10,5,10000000,3,140000,1116466,0.932157,0.932146,17.2,4.1,NaN
21,700000,"epoch=5, lr=0.07, minCount=3, wordNgrams=3",0.07,10,5,10000000,3,140000,225252,0.932143,0.932134,13.4,3.3,NaN
24,700000,"bucket=10000000, dim=10, epoch=5, lr=0.07, min...",0.07,10,5,10000000,3,140000,95590,0.932086,0.932076,14.9,4.2,9.0


## 4. Final FastText — reload and score

The cached final models are scored on the shared test corpus. This section does not retrain a model when its binary and configuration stamp already match.


In [12]:
# STEP 10 — TEST (held-out test.txt, all three sizes at FINAL)
# fit_or_load hits the cache written by step 9, so this costs no extra fits.
TEST_CSV = RESULTS_DIR / "fasttext_test_results.csv"
FINAL_LABEL = ", ".join(f"{k}={v}" for k, v in sorted(FINAL.items()))
ft_test = {}

for size in SIZES:
    model, _ = fit_or_load(size, FINAL)
    y, yhat, secs = evaluate(model, FT_DIR / "test.txt")
    acc = accuracy_score(y, yhat)
    f1  = f1_score(y, yhat, average="macro")
    val = ft_sweep.get((size, FINAL_LABEL), {}).get("validation_accuracy")
    print(f"{size:,}: test acc={acc:.6f} macroF1={f1:.6f} ({secs:.1f}s)")

    ft_test[size] = dict(
        sample_size=size,
        lr=FINAL["lr"], epoch=FINAL["epoch"], dim=FINAL["dim"],
        wordNgrams=FINAL["wordNgrams"], minCount=FINAL["minCount"],
        test_accuracy=acc, test_f1=f1, seconds=round(secs, 1),
        validation_accuracy=val,
        val_minus_test=(val - acc) if val is not None else None,
    )
    del model, y, yhat; gc.collect()

pd.DataFrame(ft_test.values()).to_csv(TEST_CSV, index=False)
pd.DataFrame(ft_test.values())


700,000: test acc=0.931220 macroF1=0.931220 (10.9s)
1,500,000: test acc=0.937075 macroF1=0.937075 (10.1s)
3,600,000: test acc=0.941965 macroF1=0.941965 (11.3s)


,sample_size,lr,epoch,dim,wordNgrams,minCount,test_accuracy,test_f1,seconds,validation_accuracy,val_minus_test
0,700000,0.07,5,10,3,9,0.931220,0.931220,10.9,0.932086,0.000866
1,1500000,0.07,5,10,3,9,0.937075,0.937075,10.1,0.937830,0.000755
2,3600000,0.07,5,10,3,9,0.941965,0.941965,11.3,0.943404,0.001439


## Headline

The saved full-data run achieved **94.34% validation accuracy** and **94.20% test accuracy** using `wordNgrams=3`, `dim=10`, and `minCount=9`.


## Reproducibility and vocabulary pruning

The optional checks measure variation across repeated eight-thread runs and compare `minCount` values. They remain disabled unless `RUN_EXPERIMENTS=True`.


In [13]:
# STEP 11 — noise floor: identical config, identical seed, 3 runs
# only thread scheduling varies. Sets the error bar: any gap smaller than the
# spread is noise. minCount pinned to 1 so this matches the recorded result.
if RUN_EXPERIMENTS:
    reps = []
    for i in range(3):
        m = fasttext.train_supervised(input=str(FT_DIR / "train_700000.txt"),
                                      **{**FT_PARAMS, "lr": 0.07, "epoch": 5,
                                         "wordNgrams": 3, "minCount": 1})
        y, yhat, _ = evaluate(m, FT_DIR / "val_700000.txt")
        reps.append(accuracy_score(y, yhat))
        print(f"rep {i+1}: {reps[-1]:.6f}")
        del m; gc.collect()
    print(f"spread = {max(reps) - min(reps):.6f}")


In [14]:
# STEP 12 — minCount: prunes rare words from the vocab
if RUN_EXPERIMENTS:
    GRID = [dict(lr=0.07, epoch=5, wordNgrams=3, minCount=mc) for mc in (1, 3, 6, 9)]
    for over in GRID:
        sweep(TUNE_SIZE, over)


## Runtime Considerations with Limited Resources

- **Training size:** Accuracy increased from **93.21% at 700k** to **94.34% at 3.6M**, while fitting time increased from **14.9 to 75.3 seconds**.

- **Memory:** The full model was **415.6 MB** and used approximately **474 MiB** when loaded.

- **Vocabulary:** `minCount=9` reduced vocabulary from **3.63M to 271k words** with negligible accuracy loss.

- **N-grams:** Trigrams improved full-data accuracy from **94.07% to 94.34%**, but increased fitting time from **54.5 to 93.7 seconds**.

- **Limited resources:** Tune using 700k reviews, process the full CSV in chunks, and train the selected configuration only once on the full dataset.

## Borderline errors

- 111401: Short positive review confused by the -ve phrasing  “Why do I have to…”.
- 163624: Positive overall, but contains several negative criticisms sbouy the book, the model focuses on the wrong subject
- 268593: Positive title and opening words conflict with a strongly negative body that is not about product
- 132276: Clear negative wording and negation “DO NOT LIKE” but predicted positive. Must be mislabeled.

In [17]:
rows_to_show = [111401, 163624, 268593, 132276]

borderline_showcase = wrong.loc[rows_to_show].reset_index()

with pd.option_context("display.max_colwidth", None):
    display(borderline_showcase[
        ["row", "error_type", "confidence", "title", "text"]
    ])

,row,error_type,confidence,title,text
0,111401,positive → negative,0.500011,Dance your shorts off!,"I enjoyed this CD. Why do i have to write 14 words , if all i have to say is that i liked the product?"
1,163624,positive → negative,0.500017,All that & Then Some,"This was a real good book to rea, but the last story left a great deal out, Adrian Quant was a caring man, but when he came across his sister he did nothing to help her or to make her that he was sorry or anything on the human side, because early in the book he did not were she was at. Other then this was great for new author."
2,268593,negative → positive,0.500214,good response from keyboard and mouse but...,typing is not fun. the keys are very stiff. typing is tough. ONLY buy this keyboard if you are not gonna use it for a lot of typing. i gotta go ice my hands now.
3,132276,negative → positive,0.500300,I DO NOT LIKE THIS ALBUM!!!,"I held out for months to buy this album hoping the price would be reduced. No go on that. All that waiting only to find out that i DO NOT LIKE IT! Soon after listening, I caught an attitude. I was mad that I purchased this. I enjoyed her other albums immensely. I cannot say that i like every song on this album. Some will make the Ipod. I can appreciate the 'ol school feel of the music, but I think it was overkill. I don't know, I just wasn't happy with this one."


## High Confidence errors

- 397676: More to do with delivery than product.
- 80534: Clearly positive recommendation labelled negative, mislabled or noise.
- 341440: Less about the product more about the fact that buying used was a bad choice.

In [ ]:
rows_to_show = [397676, 80534, 341440]

high_confidence_showcase = wrong.loc[rows_to_show].reset_index()

with pd.option_context("display.max_colwidth", None):
    display(high_confidence_showcase[
        ["row", "error_type", "confidence", "title", "text"]
    ])

,row,error_type,confidence,title,text
0,397676,positive → negative,1.0,Damaged package,"I was very disappointed when my package arrived. The bottle had leaked all in the box, therefore I did not receive the full bottle. It appeared the bottle cap was loosened."
1,80534,negative → positive,1.0,Iguanas,This book was very helpful in learning about the do's and don'ts of having a pet iguana. The information was very clearly presented. I recommend this book to new iguana owners.
2,14250,negative → positive,1.0,Easy Riffs,This series of books contains easy riffs - but don't spend much time or money locating them.
3,341440,positive → negative,1.0,"DON""T BUY USED!","I bought this used, which was perhaps a mistake- by the time it got to me the Dairy Fairy had long since drowned and her wings just didn't taste the same after that."
